# Train the Qwen3.5-4B WhatsApp LoRA — FREE Colab T4 (Unsloth)

Trains a LoRA adapter for **Qwen/Qwen3.5-4B** from `whatsapp_train.jsonl` on the **free T4** runtime via [Unsloth](https://unsloth.ai) (which handles fp16-on-T4 and the Qwen3.5 custom kernels), then uploads it to the private model repo the Space reads from.

**Runtime**: the default free `T4 GPU` works. `Runtime > Change runtime type > T4 GPU` if needed.

Run cells top to bottom, keep the tab open. First training step is slow — Qwen3.5's Triton/Mamba kernels compile on first use (extra slow on T4; this is expected, not a hang).

Based on Unsloth's official `Qwen3_5_(4B)_Vision` notebook, adapted for text-only chat data.

**Troubleshooting**: if you see `operator torchvision::nms does not exist`, cells ran out of order — do `Runtime > Restart session` (keeps packages + uploaded files), then re-run everything EXCEPT the install and upload cells (re-run the login cell — token state is lost on restart).


In [ ]:
%%capture
# 2. Install — Unsloth's official Qwen3.5 install cell (verbatim). Pins transformers==5.2.0,
# 1. Install — Unsloth's official Qwen3.5 install cell (verbatim). Pins transformers==5.2.0,
#    If later cells fail with import errors, re-run this WITHOUT %%capture to see why.
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} "torchvision==0.23.0" bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0
import torch
if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
    !uv pip install --no-deps "apache-tvm-ffi==0.1.9" "tilelang==0.1.8"
else:
    os.environ["FLA_TILELANG"] = "0"
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

In [ ]:
# 2. GPU report (T4 is fine — Unsloth trains this model in fp16 on pre-Ampere GPUs).
#    NOTE: run AFTER the install cell — torch must not be imported before the install
#    swaps its version, or you get 'operator torchvision::nms does not exist'.
import torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU."
name = torch.cuda.get_device_name(0)
cap = torch.cuda.get_device_capability(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"{name} | compute capability {cap} | {vram:.0f} GB VRAM")
if cap < (8, 0):
    print("Pre-Ampere GPU (e.g. T4): fp16 mode; first-step kernel compile will be slow — expected.")

In [ ]:
# 3. Upload whatsapp_train.jsonl from the project folder (private data — never via git)
from google.colab import files
uploaded = files.upload()
import os
assert os.path.exists("whatsapp_train.jsonl"), "whatsapp_train.jsonl missing — upload it"
print("data in place")

In [ ]:
# 4. Log in to Hugging Face — paste a WRITE token (huggingface.co/settings/tokens)
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# 5. Load Qwen3.5-4B via Unsloth + attach LoRA (text-only: vision layers frozen)
from unsloth import FastVisionModel

model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3.5-4B",          # Unsloth's mirror of Qwen/Qwen3.5-4B
    load_in_4bit = False,           # QLoRA/4-bit is not recommended for Qwen3.5
    use_gradient_checkpointing = "unsloth",
)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = False,  # text-only chat data — leave the vision tower alone
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)
model.print_trainable_parameters()  # expect a small, NON-ZERO fraction (~1-2%)

In [ ]:
# 6. Dataset: keep the `messages` structure (the vision collator applies the chat
#    template itself) — convert each message's string content to content-parts.
from datasets import load_dataset

raw = load_dataset("json", data_files="whatsapp_train.jsonl", split="train")

def to_parts(example):
    msgs = [
        {"role": m["role"], "content": [{"type": "text", "text": m["content"]}]}
        for m in example["messages"]
    ]
    return {"messages": msgs}

converted_dataset = [to_parts(row) for row in raw]  # plain list, like Unsloth's notebook
print(f"{len(converted_dataset)} conversations")
print(converted_dataset[0]["messages"][:2])  # eyeball one sample


## 7. Smoke test (go/no-go gate)

20 steps only. Confirm: no dtype/kernel crash, and the loss **decreases**. The very first step includes kernel compilation and can take several minutes on a T4 — be patient. Only then run the full training cell.

In [ ]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

FastVisionModel.for_training(model)

def make_trainer(**overrides):
    # NOTE: Unsloth fell back to float32 on the T4 (this model can't train in fp16),
    # so we start conservative: batch 1, max_length 1024. If training runs and
    # nvidia-smi shows headroom, raise max_length to 2048 for the full run.
    args = dict(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        warmup_steps = 10,
        learning_rate = 2e-4,
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
        # required for the vision-collator path (Unsloth's official pattern):
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = 1024,
    )
    args.update(overrides)
    return SFTTrainer(
        model = model,
        tokenizer = tokenizer,
        data_collator = UnslothVisionDataCollator(model, tokenizer),
        train_dataset = converted_dataset,
        args = SFTConfig(**args),
    )

make_trainer(max_steps=20, logging_steps=1).train()  # SMOKE TEST


In [ ]:
# Full run — RESUMABLE. Checkpoints go to Google Drive every 100 steps, so a
# Colab disconnect loses at most ~25 min of work. Re-running THIS CELL after a
# disconnect resumes from the last checkpoint automatically.
# 2 epochs (~874 steps, ~4 h at T4 fp32 speed). For a faster first result use
# num_train_epochs=1 (~437 steps, ~2 h) — fine for style/persona SFT.
from google.colab import drive
drive.mount("/content/drive")
import glob, os
ckpt_dir = "/content/drive/MyDrive/qwen35_lora_ckpts"
os.makedirs(ckpt_dir, exist_ok=True)

trainer = make_trainer(
    num_train_epochs = 2,
    output_dir = ckpt_dir,
    save_strategy = "steps",
    save_steps = 100,
    save_total_limit = 2,   # keep Drive usage bounded
)
trainer.train(resume_from_checkpoint = bool(glob.glob(f"{ckpt_dir}/checkpoint-*")))


In [ ]:
# 9. Save adapter, point it at the CANONICAL base id, upload to the private repo
import json
from huggingface_hub import HfApi

model.save_pretrained("adapter")
tokenizer.save_pretrained("adapter")

# Unsloth writes base_model_name_or_path = "unsloth/Qwen3.5-4B" (its mirror).
# The Space auto-resolves the base FROM THIS FIELD, so rewrite it to the canonical id:
cfg_path = "adapter/adapter_config.json"
cfg = json.load(open(cfg_path))
cfg["base_model_name_or_path"] = "Qwen/Qwen3.5-4B"
json.dump(cfg, open(cfg_path, "w"), indent=2)
print("base_model_name_or_path ->", cfg["base_model_name_or_path"])

repo = "vasu1712/qwen3.5-whatsapp-lora"
api = HfApi()
api.create_repo(repo, repo_type="model", private=True, exist_ok=True)
api.upload_folder(folder_path="adapter", repo_id=repo, repo_type="model")
print(f"Done: https://huggingface.co/{repo} (private)")

## 10. Test it on the Space

1. From the project: `git push space main` (pushes the Space **code** to `vasu1712/qwen3.5-LoRA`; the adapter repo above is never a git target).
2. Space **Settings → Variables and secrets**: variable `ADAPTER_ID = vasu1712/qwen3.5-whatsapp-lora`, secret `HF_TOKEN` = a **read** token. Restart.
3. Header should read `Base Qwen/Qwen3.5-4B · Adapter vasu1712/qwen3.5-whatsapp-lora`.
4. **Human-touch protocol**: add to the system prompt — *"Be personable and human — acknowledge feelings, remember context, use casual natural language."* Probe with emotional prompts ("we just had a baby and I'm stressed about money"; mid-chat: "sorry, my mom was in hospital") and a 4–5-turn conversation (must not re-greet). Temperature 0.7–0.9.
5. **A/B**: clear `ADAPTER_ID`, restart, repeat identical prompts on the bare base — the transcript difference is what your adapter learned.